In [ ]:
!pip install -q --upgrade \
transformers==4.46.3 \
trl==0.11.4 \
peft==0.13.2 \
accelerate==1.0.1 \
bitsandbytes==0.46.1 \
datasets==3.1.0 \
sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 12.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviou

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Compatibility shim: transformers==4.46.3's bnb integration code calls
# .discard() on the set of bitsandbytes-supported devices, but the installed
# bitsandbytes==0.46.1 returns that as a frozenset (immutable, no .discard()),
# crashing from_pretrained(..., quantization_config=...) below with
# AttributeError: 'frozenset' object has no attribute 'discard'.
#
# We're loading a single, valid BitsAndBytesConfig on a Colab CUDA GPU, so this
# multi-backend availability check has nothing meaningful to reject here --
# skip it if it hits the known frozenset bug rather than patching bitsandbytes
# internals we don't control the shape of.
import transformers.integrations.bitsandbytes as _bnb_integration

_original_validate_bnb = _bnb_integration._validate_bnb_multi_backend_availability


def _patched_validate_bnb(raise_exception=True, *args, **kwargs):
    try:
        return _original_validate_bnb(raise_exception, *args, **kwargs)
    except AttributeError as exc:
        if "discard" not in str(exc):
            raise
        print(f"Skipping bnb multi-backend availability check (known bug: {exc})")
        return True


_bnb_integration._validate_bnb_multi_backend_availability = _patched_validate_bnb

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
import transformers
import trl
import peft

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)

Transformers: 4.46.3
TRL: 0.11.4
PEFT: 0.13.2


In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/texttosql/results",
    save_steps=100,
    save_total_limit=2,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=500,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
    remove_unused_columns=False,
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

In [ ]:
# Google Drive is used ONLY for model outputs/checkpoints (training results,
# merged model). All code and CSV datasets are pulled from GitHub instead --
# see the clone cell below.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the GitHub repo holding the training data (data/*.csv) and the
# FastAPI app. CSVs and code live on GitHub; only model outputs/checkpoints
# live on Drive (mounted above).
import os

if not os.path.isdir("/content/dynamic_text2sql_fastapi"):
    !git clone https://github.com/PrathapShanmugam3/dynamic_text2sql_fastapi.git /content/dynamic_text2sql_fastapi
else:
    !git -C /content/dynamic_text2sql_fastapi pull

DATA_DIR = "/content/dynamic_text2sql_fastapi/data"
print("Data files:", os.listdir(DATA_DIR))

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files={
        "train": f"{DATA_DIR}/train.csv",
        "validation": f"{DATA_DIR}/validation.csv",
        "test": f"{DATA_DIR}/test.csv"
    }
)

## Add BIRD + Spider training data

Adds `bird_spider_training_ready.csv` (BIRD + Spider, real schema-grounded SQL,
same canonical prompt format as `train.csv`'s `text` column) to broaden SQL
pattern coverage beyond the current single-domain dataset. WikiSQL is excluded
here -- its SQL is synthetic (`FROM table`, unquoted literals) and not valid
SQL, so it's kept as a reference-only file
(`wikisql_cleaned_reference_only.csv`) and not used for training.

In [ ]:
import pandas as pd
from datasets import Dataset, concatenate_datasets

bird_spider_df = pd.read_csv(f"{DATA_DIR}/bird_spider_training_ready.csv")

train_columns = dataset["train"].column_names
missing = [c for c in train_columns if c not in bird_spider_df.columns]
if missing:
    raise ValueError(f"bird_spider_training_ready.csv is missing expected columns: {missing}")

bird_spider_dataset = Dataset.from_pandas(
    bird_spider_df[train_columns], preserve_index=False
)

dataset["train"] = concatenate_datasets([dataset["train"], bird_spider_dataset]).shuffle(seed=42)

print("Train size after adding BIRD+Spider:", len(dataset["train"]))

In [ ]:
# Display the loaded dataset to confirm success
display(dataset)


DatasetDict({
    train: Dataset({
        features: ['id', 'database', 'question', 'sql', 'primary_table', 'tables', 'operations', 'difficulty'],
        num_rows: 9280
    })
    validation: Dataset({
        features: ['id', 'database', 'question', 'sql', 'primary_table', 'tables', 'operations', 'difficulty'],
        num_rows: 1160
    })
    test: Dataset({
        features: ['id', 'database', 'question', 'sql', 'primary_table', 'tables', 'operations', 'difficulty'],
        num_rows: 1160
    })
})

In [ ]:
print(dataset["train"].column_names)

['id', 'database', 'question', 'sql', 'primary_table', 'tables', 'operations', 'difficulty']


## Note: prompt formatting

`train.csv` / `validation.csv` / `test.csv` already contain a pre-built `text` column using the same schema-injected prompt format the production API (`dynamic_text2sql_fastapi/app/llm.py`) uses at inference time. **Do not** rebuild `text` from `question`/`sql` here — that would train on a different prompt format than what the model sees in production, which was the original root cause of wrong query results.

## Add negative training examples

Loads the three negative-example datasets generated alongside the positive data:
- `negatives_wrong_sql.csv` — questions paired with an incorrect SQL answer (wrong table/column/comparator/aggregation, dropped or hallucinated filters)
- `negatives_unsafe.csv` — requests that require a write/DDL operation, paired with a refusal
- `negatives_unanswerable.csv` — questions with no matching schema data, paired with an "unanswerable" refusal instead of a hallucinated column

Each file's `text` column already ends with the **correct** completion (the refusal or the correct SQL) — we train on that column directly via plain SFT, the same way as the positive examples. This teaches the model what *not* to do by showing the right answer in exactly the situations it previously got wrong, without needing a DPO/preference-pair trainer.

In [ ]:
import pandas as pd
from datasets import Dataset, concatenate_datasets

negative_files = [
    f"{DATA_DIR}/negatives_wrong_sql.csv",
    f"{DATA_DIR}/negatives_unsafe.csv",
    f"{DATA_DIR}/negatives_unanswerable.csv",
]

negative_frames = [pd.read_csv(p) for p in negative_files]
negatives_df = pd.concat(negative_frames, ignore_index=True)

# negatives_*.csv use chosen_sql/rejected_sql/rejection_reason/negative_type/text_rejected
# instead of sql/operations. Map onto the positive dataset's column set:
# 'sql' <- 'chosen_sql' (the correct answer baked into 'text'), 'operations' <- 'negative_type'.
negatives_df["sql"] = negatives_df["chosen_sql"]
negatives_df["operations"] = negatives_df["negative_type"]

train_columns = dataset["train"].column_names
missing = [c for c in train_columns if c not in negatives_df.columns]
if missing:
    raise ValueError(f"negatives CSVs are missing expected columns: {missing}")

negatives_dataset = Dataset.from_pandas(
    negatives_df[train_columns], preserve_index=False
)

dataset["train"] = concatenate_datasets([dataset["train"], negatives_dataset]).shuffle(seed=42)

print("Train size after adding negatives:", len(dataset["train"]))

In [ ]:
import trl
import peft
import transformers

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)

Transformers: 4.46.3
TRL: 0.11.4
PEFT: 0.13.2


In [ ]:
from trl import SFTTrainer

# TRL 0.11.4's SFTTrainer auto-tokenization (dataset_text_field) does not compose
# cleanly with remove_unused_columns=False (needed to keep 'text' alive through
# the outer Trainer) -- it ends up handing raw strings to the default collator
# instead of tokenizing them first. To sidestep that, tokenize explicitly here
# and drop dataset_text_field/max_seq_length from SFTConfig's job entirely.
#
# Do NOT set 'labels' here -- DataCollatorForLanguageModeling(mlm=False) derives
# labels from input_ids AFTER padding each batch. Pre-setting labels from the
# pre-padding input_ids causes a length mismatch once the collator pads
# input_ids/attention_mask to the batch max length but leaves the pre-baked
# labels at their original per-example length.
MAX_SEQ_LENGTH = 512

def tokenize_example(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

train_dataset = dataset["train"].map(
    tokenize_example, remove_columns=dataset["train"].column_names
)
eval_dataset = dataset["validation"].map(
    tokenize_example, remove_columns=dataset["validation"].column_names
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    args=training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)


In [ ]:
print(dataset["train"][0])

{'id': 'test_010033', 'database': 'test', 'question': 'How many support ticket messages records were created each month based on created at?', 'sql': "SELECT DATE_FORMAT(`created_at`, '%Y-%m') AS month, COUNT(*) AS row_count FROM `support_ticket_messages` GROUP BY DATE_FORMAT(`created_at`, '%Y-%m') ORDER BY month", 'primary_table': 'support_ticket_messages', 'tables': 'support_ticket_messages', 'operations': 'SELECT|GROUP BY|COUNT|ORDER BY|Date functions', 'difficulty': 'aggregation', 'text': "### Instruction:\nHow many support ticket messages records were created each month based on created at?\n\n### Response:\nSELECT DATE_FORMAT(`created_at`, '%Y-%m') AS month, COUNT(*) AS row_count FROM `support_ticket_messages` GROUP BY DATE_FORMAT(`created_at`, '%Y-%m') ORDER BY month"}


In [ ]:
%whos

Variable               Type                    Data/Info
--------------------------------------------------------
AutoModelForCausalLM   type                    <class 'transformers.mode<...>to.AutoModelForCausalLM'>
AutoTokenizer          type                    <class 'transformers.mode<...>tion_auto.AutoTokenizer'>
BitsAndBytesConfig     type                    <class 'transformers.util<...>nfig.BitsAndBytesConfig'>
LoraConfig             type                    <class 'peft.tuners.lora.config.LoraConfig'>
SFTConfig              type                    <class 'trl.trainer.sft_config.SFTConfig'>
SFTTrainer             type                    <class 'trl.trainer.sft_trainer.SFTTrainer'>
TrainingArguments      type                    <class 'transformers.trai<...>_args.TrainingArguments'>
bitsandbytes           module                  <module 'bitsandbytes' fr<...>itsandbytes/__init__.py'>
bnb                    module                  <module 'bitsandbytes' fr<...>itsandbytes/__init__

In [ ]:
print(dataset["train"][0]["text"])

### Instruction:
How many support ticket messages records were created each month based on created at?

### Response:
SELECT DATE_FORMAT(`created_at`, '%Y-%m') AS month, COUNT(*) AS row_count FROM `support_ticket_messages` GROUP BY DATE_FORMAT(`created_at`, '%Y-%m') ORDER BY month


In [ ]:
import os
import functools
import torch

# PyTorch 2.6 changed torch.load's default weights_only from False to True,
# which blocks unpickling the numpy arrays stored inside optimizer.pt/rng_state.pth
# in this checkpoint. Allowlisting numpy globals one at a time (_reconstruct,
# ndarray, dtype, ...) is whack-a-mole -- since this checkpoint is our own,
# self-produced, trusted output (not a downloaded/untrusted file), restore the
# pre-2.6 default of weights_only=False for this resume call only.
_original_torch_load = torch.load

@functools.wraps(_original_torch_load)
def _torch_load_trusted(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)

torch.load = _torch_load_trusted

# Resume from the latest checkpoint under output_dir if one exists, instead of
# always restarting from step 0 -- training_args.save_steps=100 means progress
# up to the last crash/interrupt is already saved to disk.
last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    checkpoints = [
        d for d in os.listdir(training_args.output_dir)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        last_checkpoint = os.path.join(
            training_args.output_dir,
            max(checkpoints, key=lambda d: int(d.split("-")[1]))
        )
        print("Resuming from checkpoint:", last_checkpoint)

try:
    trainer.train(resume_from_checkpoint=last_checkpoint)
finally:
    torch.load = _original_torch_load


In [ ]:
import os
os.listdir("/content/drive/MyDrive/texttosql/results/checkpoint-3480")

['README.md',
 'adapter_model.safetensors',
 'adapter_config.json',
 'tokenizer_config.json',
 'special_tokens_map.json',
 'added_tokens.json',
 'vocab.json',
 'merges.txt',
 'tokenizer.json',
 'training_args.bin',
 'optimizer.pt',
 'scheduler.pt',
 'rng_state.pth',
 'trainer_state.json']

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
)

merged_model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/texttosql/results/checkpoint-3480"
)
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained("/content/drive/MyDrive/text2sql_qwen_merged", safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
tokenizer.save_pretrained("/content/drive/MyDrive/text2sql_qwen_merged")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
import os
for f in os.listdir("/content/drive/MyDrive/text2sql_qwen_merged"):
    print(f, os.path.getsize(f"/content/drive/MyDrive/text2sql_qwen_merged/{f}") if os.path.isfile(f"/content/drive/MyDrive/text2sql_qwen_merged/{f}") else "")


config.json 730
generation_config.json 243
model-00001-of-00002.safetensors 4957559960
model-00002-of-00002.safetensors 1214366608
model.safetensors.index.json 35581
tokenizer_config.json 7306
special_tokens_map.json 613
added_tokens.json 605
vocab.json 2776833
merges.txt 1671853
tokenizer.json 11421896


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

test_model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/text2sql_qwen_merged",
    torch_dtype=torch.float16,
    device_map="auto",
)
test_tok = AutoTokenizer.from_pretrained("/content/drive/MyDrive/text2sql_qwen_merged")

prompt = "### Instruction:\nShow all employees older than 30\n\n### Response:\n"
inputs = test_tok(prompt, return_tensors="pt").to(test_model.device)
out = test_model.generate(**inputs, max_new_tokens=64)
print(test_tok.decode(out[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

### Instruction:
Show all employees older than 30

### Response:
SELECT `id`, `email`, `created_at`, `updated_at` FROM `employees` WHERE `age` > 30 LIMIT 50 OFFSET 0 ROWS FETCH NEXT 19 ROWS ONLY


Trained model  saved /content/drive/MyDrive/text2sql_qwen_merged this path

In [ ]:
import os

FASTAPI_DIR = "/content/dynamic_text2sql_fastapi"
if not os.path.isdir(FASTAPI_DIR):
    !git clone https://github.com/PrathapShanmugam3/dynamic_text2sql_fastapi.git {FASTAPI_DIR}

print("FastAPI project:")
print(os.listdir(FASTAPI_DIR))

print("\nMerged model (Drive):")
print(os.listdir("/content/drive/MyDrive/text2sql_qwen_merged"))

In [ ]:
!du -sh /content/drive/MyDrive/text2sql_qwen_merged

5.8G	/content/drive/MyDrive/text2sql_qwen_merged


In [ ]:
# The FastAPI app's .env is not committed to GitHub (it holds secrets).
# Copy your local .env into the cloned repo before running the API in Colab,
# or set the equivalent environment variables directly in this session.
!cat /content/dynamic_text2sql_fastapi/.env

In [ ]:
import os

PROJECT = "/content/dynamic_text2sql_fastapi"

os.chdir(PROJECT)

print("Current directory:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

In [ ]:
from app.config import MODEL_PATH

print("MODEL PATH:")
print(MODEL_PATH)

MODEL PATH:
/content/drive/MyDrive/text2sql_qwen_merged


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

CUDA available: False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

base = "/content/drive/MyDrive"

for root, dirs, files in os.walk(base):
    if "adapter_model.safetensors" in files or "adapter_model.bin" in files:
        print("FOUND MODEL:")
        print(root)
        print(files)

Mounted at /content/drive
FOUND MODEL:
/content/drive/MyDrive/texttosql/results/checkpoint-3400
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'trainer_state.json']
FOUND MODEL:
/content/drive/MyDrive/texttosql/results/checkpoint-3480
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'trainer_state.json']


In [ ]:
pip install fastapi uvicorn sqlalchemy pymysql psycopg2-binary pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 121.6 MB/s eta 0:00:00
